# 22 · Pick the right value from extracted candidates

## The problem

An invoice email mentions **three dates** and **four amounts**. Which one is the *due date*? Which is the *total due*?

- **Regex** finds every date and amount easily, but it can't tell the order date from the due date.
- **An LLM extractor** can tell them apart, but it *generates* the answer, so it can also reformat it, round it, or
  invent a value that isn't in the email at all. For invoices, payments and contracts, that's not acceptable.

## Why Jev

Split the job in two. **Code extracts** the literal candidate spans with a regex. **Jev only chooses** among them
with a `Choice`, plus a `none` option for "none of these is the right answer". The chosen span is then copied out of
the email **unchanged**. The output can never contain a value that isn't in the source, by construction, and every
selection comes with a calibrated probability. It's a System One job: one typed call, about 100 ms, sub-cent.

## What we'll build

```
email ──► regex: candidate dates + amounts ──► Jev: one Choice per field ──► copy the chosen span verbatim
           ['3 March 2025', '2 April 2025', ...]    due_date  ∈ {d0, d1, d2, none}     due_date  = '2 April 2025'
           ['$1,200.00', '$1,296.00', ...]           total_due ∈ {a0, a1, a2, a3, none} total_due = '$1,296.00'
```

## What you'll learn
- Jev as a **selector, not a generator**: grounded extraction where hallucinated values are impossible.
- How to put **several `Choice` questions in one call**, each with its own options, over a dict `state`.
- Why a `none` option matters: an email with no due date should return `None`, not the closest-looking date.

*From the use-case catalog: [18 practical Jev use cases for AI agents](https://huggingface.co/blog/karmen-beatapi/18-practical-jev-use-cases-for-ai-agents), #16.*

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell loads `TYPESAFE_API_KEY` from the `.env` file at the repo root, creates the Jev client `jev`, and defines
`bar()`, a small helper that draws a 0..1 probability as a text bar. We only need the **`Choice`** question type here.

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()


def bar(p, width=20):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · Extract candidates with plain regex

Two regular expressions, deliberately **greedy about recall**: they should catch every date and amount, even the ones
that aren't the answer. Telling them apart is Jev's job, not the regex's.

- `DATE_RE` matches `2 April 2025`-style dates and ISO dates like `2025-06-01`.
- `MONEY_RE` matches `$`, `€`, `£` or `USD` amounts, with optional thousands separators, cents, and a `k`/`M` suffix.

> **Workshop note:** the original `MONEY_RE` had no `k`/`M` suffix, so *"the $2M funding news"* produced the candidate
> `$2`, a span that misquotes the email. Grounding is only as good as your candidates: the regex must capture the
> whole span. Adding `(?:[kKmM]\b)?` fixed it.

In [2]:
DATE_RE = r"\b(?:\d{1,2} (?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{4}|\d{4}-\d{2}-\d{2})\b"
MONEY_RE = r"(?:USD|\$|€|£)\s?\d[\d,]*(?:\.\d{2})?(?:[kKmM]\b)?"

EMAIL = """Hi team, thanks for your order on 3 March 2025.
Invoice #INV-2291 is attached. The subtotal is $1,200.00, plus $96.00 tax, for a total of $1,296.00.
Please pay by 2 April 2025. A late fee of $50.00 applies after that date.
Our next maintenance window is 15 April 2025."""

dates = re.findall(DATE_RE, EMAIL)
amounts = re.findall(MONEY_RE, EMAIL)
print("date candidates  :", dates)
print("amount candidates:", amounts)

date candidates  : ['3 March 2025', '2 April 2025', '15 April 2025']
amount candidates: ['$1,200.00', '$96.00', '$1,296.00', '$50.00']


**What to notice**
- Three dates (order date, due date, maintenance window) and four amounts (subtotal, tax, total, late fee).
- The regex has no idea which is which. That's fine: it only has to make sure the right answer is *somewhere* in the list.

## Step 2 · Jev chooses among the candidates

`FIELDS` maps each field we want to the candidate list it picks from and the question to ask.

`select_fields()` then builds **one `Choice` per field** and sends them all in **one** `system_one` call:

- **Option keys are opaque** (`due_date_0`, `due_date_1`, ...) and each **description is the literal span**
  (`"2 April 2025"`). Jev picks a key; `keymap` turns it back into the exact text from the email.
- Every question also gets a `none` option: *"None of these candidates"*. If Jev picks it, the field is `None`.
- **`state` is a dict** here: `{"email": text}`. In the instructions, `` `email` `` (in backticks) refers to that key,
  so Jev knows exactly which input the question is about. That matters once you pass several inputs.
- From each `Choice` answer we keep `.choice` (the winning key), `.confidence`, and `.probabilities["none"]`, so we can
  see how strongly Jev rejected every candidate.

This cell only **defines** things. The call happens in the next cell.

In [3]:
FIELDS = {
    "due_date": ("dates", "Which of these is the date by which the invoice must be paid?"),
    "total_due": ("amounts", "Which of these is the total amount the customer must pay?"),
}


def select_fields(text, client):
    """Return ({field: {"value", "conf", "p_none"}}, candidates). Values are copied verbatim from `text`, or None."""
    if "FIELDS" not in globals():
        raise RuntimeError("FIELDS is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    cands = {"dates": re.findall(DATE_RE, text), "amounts": re.findall(MONEY_RE, text)}
    questions, keymap = {}, {}
    for field, (kind, ask) in FIELDS.items():
        if not cands[kind]:                       # no candidates at all: nothing to ask, the field stays None
            continue
        opts = {f"{field}_{i}": span for i, span in enumerate(cands[kind])}
        keymap.update(opts)
        questions[field] = Choice(instructions=f"In `email`, {ask[0].lower() + ask[1:]}",
                                  criteria={**opts, "none": "None of these candidates"})

    out = {f: {"value": None, "conf": 0.0, "p_none": 1.0} for f in FIELDS}
    if questions:
        r = client.system_one(state={"email": text}, questions=questions)
        for field in questions:
            c = r.choices[field]
            out[field] = {"value": keymap.get(c.choice),      # copied verbatim, or None if Jev chose "none"
                          "conf": c.confidence,
                          "p_none": c.probabilities.get("none", 0.0)}
    return out, cands

### Run it on the invoice email

In [4]:
fields, cands = select_fields(EMAIL, jev)
for f, v in fields.items():
    print(f"{f:<10} = {v['value']!r:<16} confidence {v['conf']:.2f}   p(none) {v['p_none']:.2f}")

due_date   = '2 April 2025'   confidence 1.00   p(none) 0.00
total_due  = '$1,296.00'      confidence 1.00   p(none) 0.00


**What to notice**
- `due_date` is the *"please pay by"* date, not the order date or the maintenance window.
- `total_due` is the total including tax, not the subtotal, the tax line or the late fee.
- Both answers are near-certain and `p(none)` is near 0: Jev is sure a right answer is among the candidates.
- One request answered both questions.

## Step 3 · Prove it can't invent values

This is the property that makes the pattern safe: every selected value must be a **literal substring** of the source.
With a generative extractor you'd *hope* that holds. Here it's guaranteed by the code, and this `assert` checks it.

In [5]:
for f, v in fields.items():
    assert v["value"] is None or v["value"] in EMAIL, f"{f} not in source!"
print("✓ every extracted value appears verbatim in the email")

✓ every extracted value appears verbatim in the email


## Step 4 · Harder emails, including ones with no due date

Three emails that each test something different:

| Email | The trap |
|---|---|
| subscription renewal | the only date is a *renewal* date, and there's a second amount from last year |
| quote | the date is when the **quote expires**, not a payment due date; there's a total and a deposit |
| meeting thank-you | a date and an amount, but **nothing is owed**: both fields should be `None` |

In [6]:
EMAILS = [
    "Reminder: your subscription renews on 2025-06-01 for $49.00. Last year you paid $39.00.",
    "Quote valid until 30 May 2025. Estimated cost €4,500.00; a 10% deposit of €450.00 is due on signing.",
    "Thanks for meeting on 12 May 2025! Great to hear the $2M funding news.",
]
for e in EMAILS:
    f, c = select_fields(e, jev)
    print(e)
    print(f"   candidates: {c['dates']} {c['amounts']}")
    for name, v in f.items():
        print(f"   {name:<10} = {v['value']!r:<14} confidence {v['conf']:.2f}   p(none) {v['p_none']:.2f}")
    assert all(v["value"] is None or v["value"] in e for v in f.values())
    print()

Reminder: your subscription renews on 2025-06-01 for $49.00. Last year you paid $39.00.
   candidates: ['2025-06-01'] ['$49.00', '$39.00']
   due_date   = '2025-06-01'   confidence 0.38   p(none) 0.31
   total_due  = '$49.00'       confidence 1.00   p(none) 0.00



Quote valid until 30 May 2025. Estimated cost €4,500.00; a 10% deposit of €450.00 is due on signing.
   candidates: ['30 May 2025'] ['€4,500.00', '€450.00']
   due_date   = None           confidence 0.74   p(none) 0.87
   total_due  = '€4,500.00'    confidence 0.97   p(none) 0.01



Thanks for meeting on 12 May 2025! Great to hear the $2M funding news.
   candidates: ['12 May 2025'] ['$2M']
   due_date   = None           confidence 0.94   p(none) 0.97
   total_due  = None           confidence 0.80   p(none) 0.90



**What to notice**
- **Funding-news email:** both fields come back `None`. `12 May 2025` is a meeting date and `$2M` is someone's
  funding round. Neither is owed by anyone, and `p(none)` is high for both. A generative extractor asked for "the due
  date" would be tempted to return the only date it saw.
- **Quote:** `due_date` is `None` too. *"Valid until"* is an expiry date, not a payment deadline, and the deposit is due
  *"on signing"*, which isn't a date. `total_due` picks the full estimate.
- **Subscription:** Jev picks the renewal date as the due date, but with noticeably **lower confidence** and a real
  `p(none)`. That's honest: a renewal date is *when you'll be charged*, which is close to, but not quite, a due date. In
  production, a confidence threshold would send cases like this to review.
- Every value printed is still a verbatim span (the `assert` in the loop checks each one).

## Step 5 · Give it a face

Everything from here on is **presentation only**: a small style kit, a form wired to `select_fields()` with
`ipywidgets`, and an HTML view that highlights the chosen spans inside the email. The selection logic above is
untouched.

In [7]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:860px;">{inner}</div>'


def badge(text, color):
    return f'<span style="{BADGE}background:{color};">{text}</span>'


def meter(label, p, width=160):
    """A labelled horizontal probability bar."""
    col = RED if p > 0.7 else AMBER if p > 0.4 else SAGE
    return (f'<div style="display:flex;align-items:center;gap:8px;font-size:12px;margin:2px 0;">'
            f'<span style="width:190px;color:{SLATE};">{label}</span>'
            f'<span style="width:{width}px;height:8px;background:#ECE9E0;border-radius:4px;overflow:hidden;">'
            f'<span style="display:block;height:8px;width:{p * 100:.0f}%;background:{col};"></span></span>'
            f'<b>{p:.2f}</b></div>')


def form_app(fields, render, button="Run"):
    """fields: list of (name, label, default, rows). render(**values) returns HTML."""
    boxes = {}
    for name, label, default, rows in fields:
        w = (widgets.Textarea(value=default, rows=rows, layout=widgets.Layout(width="90%"))
             if rows > 1 else widgets.Text(value=default, layout=widgets.Layout(width="90%")))
        boxes[name] = w
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            display(HTML(box("<i>thinking...</i>")))
            html = render(**{k: w.value.strip() for k, w in boxes.items()})
            out.clear_output(wait=True)
            display(HTML(html))

    btn.on_click(go)
    rows = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), boxes[name]]) for name, label, _, _ in fields]
    display(widgets.VBox(rows + [btn]), out)

`spans_html()` highlights the chosen due date (coral) and total (green) **in place** in the email, then lists each
field with its confidence and all the candidates the regex found.

In [8]:
def highlight(text, spans):
    for label, span, col in spans:
        if span:
            text = text.replace(span, f'<mark style="background:{col};color:white;padding:1px 4px;border-radius:4px;" '
                                      f'title="{label}">{span}</mark>', 1)
    return text.replace("\n", "<br>")


def spans_html(email):
    f, cands = select_fields(email, jev)
    hl = highlight(email, [("due_date", f["due_date"]["value"], CORAL), ("total_due", f["total_due"]["value"], SAGE)])
    fields = "".join(f'<div>{badge(k, CORAL if k == "due_date" else SAGE)} <code>{v["value"]}</code> '
                     f'<span style="color:{SLATE};font-size:12px;">confidence {v["conf"]:.2f}</span></div>'
                     for k, v in f.items())
    cand = f'<div style="font-size:12px;color:{SLATE};margin-top:6px;">candidates: {cands["dates"]} {cands["amounts"]}</div>'
    return box(f'<div style="{CARD}"><div style="line-height:1.7;">{hl}</div></div><div style="{CARD}">{fields}{cand}</div>')


display(HTML(spans_html(EMAIL)))

## Step 6 · Try it yourself

Edit the email and press **Extract** (needs a live kernel). Good ones to try:
- The quote email (pre-filled) → `due_date` is `None`, `total_due` is `€4,500.00`.
- `Your invoice of £320.00 (incl. £20.00 shipping) is due 2025-07-15. Thanks for shopping with us on 2025-07-01.` →
  due date `2025-07-15`, total `£320.00`.
- `Payment of USD 1,000 received on 3 June 2025. Your balance is now $0.00.` → nothing is due: `due_date` should be `None`
  with a high `p(none)`, and `total_due` most likely picks the remaining balance, `$0.00`.
- `Please pay $75.00 next Friday.` → the regex finds no date at all, so `due_date` is `None` without even asking Jev.
  That's a limit of the candidate step, not of Jev.

In [9]:
form_app([("email", "Email", EMAILS[1], 4)], spans_html, "Extract")

Output()

## Recap and exercises

**Key ideas**
1. **Selector, not generator.** Code proposes literal spans, Jev picks one, the span is copied verbatim. Invented values are impossible by construction.
2. **Always offer `none`.** "None of these is the answer" is often the correct answer, and `p(none)` tells you how sure Jev is.
3. **Recall lives in the regex.** If the right span isn't a candidate (or is cut short, like `$2` for `$2M`), Jev can't pick it.
4. Many fields, one call: each field is its own `Choice` in the same `system_one` request.

**Exercises**
- Add a third field, `invoice_number`, with a regex like `r"#?INV-\d+"` and the question "Which of these is the invoice number?".
- Add a rule: if `conf < 0.8`, return `None` plus a `needs_review` flag. Which of the Step 4 emails gets flagged?
- Add relative dates ("next Friday", "end of month") to `DATE_RE` and see whether Jev picks them.
- Write a test email where the late fee is larger than the total, and check Jev still picks the total.